<a href="https://colab.research.google.com/github/Herrera1022/Proyecto-Kaggle/blob/main/99_modelo_solucion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# 0. Instalación de librerías
# ============================================================
!pip install -q catboost kaggle

import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from catboost import CatBoostClassifier, Pool


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 6.8 MB/s eta 0:00:00


In [2]:
# ============================================================
# 1. Configurar Kaggle en Colab
#    (Sube tu archivo kaggle.json)
# ============================================================
from google.colab import files

# Ejecuta esta celda y selecciona tu kaggle.json
uploaded = files.upload()

# Asegúrate de que el archivo se llame exactamente: kaggle.json
os.environ['KAGGLE_CONFIG_DIR'] = '/content'
!chmod 600 /content/kaggle.json


Saving kaggle.json to kaggle.json


In [3]:
# ============================================================
# 2. Descargar y descomprimir los datos de la competencia
# ============================================================
COMP_NAME = 'udea-ai-4-eng-20252-pruebas-saber-pro-colombia'
DATA_DIR = '/content/data'

os.makedirs(DATA_DIR, exist_ok=True)

# Descargar .zip con todos los archivos de la competencia
!kaggle competitions download -c $COMP_NAME -p $DATA_DIR

# Descomprimir (si el nombre cambia, el comodín *.zip lo cubre)
!unzip -o "$DATA_DIR"/*.zip -d "$DATA_DIR" > /dev/null

# Ver qué archivos quedaron
os.listdir(DATA_DIR)



  0% 0.00/29.9M [00:00<?, ?B/s]
100% 29.9M/29.9M [00:00<00:00, 1.41GB/s]


['submission_example.csv',
 'udea-ai-4-eng-20252-pruebas-saber-pro-colombia.zip',
 'train.csv',
 'test.csv']

In [4]:
# ============================================================
# 3. Cargar los datos
# ============================================================
train_path = os.path.join(DATA_DIR, 'train.csv')
test_path = os.path.join(DATA_DIR, 'test.csv')
sub_example_path = os.path.join(DATA_DIR, 'submission_example.csv')

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sub_example = pd.read_csv(sub_example_path)

print("Train shape:", train.shape)
print("Test shape:", test.shape)
train.head()



Train shape: (692500, 21)
Test shape: (296786, 20)


,ID,PERIODO_ACADEMICO,E_PRGM_ACADEMICO,E_PRGM_DEPARTAMENTO,E_VALORMATRICULAUNIVERSIDAD,E_HORASSEMANATRABAJA,F_ESTRATOVIVIENDA,F_TIENEINTERNET,F_EDUCACIONPADRE,F_TIENELAVADORA,...,E_PRIVADO_LIBERTAD,E_PAGOMATRICULAPROPIO,F_TIENECOMPUTADOR,F_TIENEINTERNET.1,F_EDUCACIONMADRE,RENDIMIENTO_GLOBAL,INDICADOR_1,INDICADOR_2,INDICADOR_3,INDICADOR_4
0,904256,20212,ENFERMERIA,BOGOTÁ,Entre 5.5 millones y menos de 7 millones,Menos de 10 horas,Estrato 3,Si,Técnica o tecnológica incompleta,Si,...,N,No,Si,Si,Postgrado,medio-alto,0.322,0.208,0.310,0.267
1,645256,20212,DERECHO,ATLANTICO,Entre 2.5 millones y menos de 4 millones,0,Estrato 3,No,Técnica o tecnológica completa,Si,...,N,No,Si,No,Técnica o tecnológica incompleta,bajo,0.311,0.215,0.292,0.264
2,308367,20203,MERCADEO Y PUBLICIDAD,BOGOTÁ,Entre 2.5 millones y menos de 4 millones,Más de 30 horas,Estrato 3,Si,Secundaria (Bachillerato) completa,Si,...,N,No,No,Si,Secundaria (Bachillerato) completa,bajo,0.297,0.214,0.305,0.264
3,470353,20195,ADMINISTRACION DE EMPRESAS,SANTANDER,Entre 4 millones y menos de 5.5 millones,0,Estrato 4,Si,No sabe,Si,...,N,No,Si,Si,Secundaria (Bachillerato) completa,alto,0.485,0.172,0.252,0.190
4,989032,20212,PSICOLOGIA,ANTIOQUIA,Entre 2.5 millones y menos de 4 millones,Entre 21 y 30 horas,Estrato 3,Si,Primaria completa,Si,...,N,No,Si,Si,Primaria completa,medio-bajo,0.316,0.232,0.285,0.294


In [5]:
# ============================================================
# 4. Revisar columnas y variable objetivo
# ============================================================
TARGET_COL = 'RENDIMIENTO_GLOBAL'
ID_COL = 'ID'

print("Columnas del train:\n", train.columns.tolist(), "\n")
print("Valores de la variable objetivo:")
print(train[TARGET_COL].value_counts())


Columnas del train:
 ['ID', 'PERIODO_ACADEMICO', 'E_PRGM_ACADEMICO', 'E_PRGM_DEPARTAMENTO', 'E_VALORMATRICULAUNIVERSIDAD', 'E_HORASSEMANATRABAJA', 'F_ESTRATOVIVIENDA', 'F_TIENEINTERNET', 'F_EDUCACIONPADRE', 'F_TIENELAVADORA', 'F_TIENEAUTOMOVIL', 'E_PRIVADO_LIBERTAD', 'E_PAGOMATRICULAPROPIO', 'F_TIENECOMPUTADOR', 'F_TIENEINTERNET.1', 'F_EDUCACIONMADRE', 'RENDIMIENTO_GLOBAL', 'INDICADOR_1', 'INDICADOR_2', 'INDICADOR_3', 'INDICADOR_4'] 

Valores de la variable objetivo:
RENDIMIENTO_GLOBAL
alto          175619
bajo          172987
medio-bajo    172275
medio-alto    171619
Name: count, dtype: int64


In [6]:
# ============================================================
# 5. Separar features y target
#    - NO usamos la columna ID como feature
# ============================================================

X = train.drop(columns=[TARGET_COL, ID_COL])
y = train[TARGET_COL]

X_test_final = test.drop(columns=[ID_COL])

print("Shape X:", X.shape)
print("Shape y:", y.shape)
print("Shape X_test_final:", X_test_final.shape)


Shape X: (692500, 19)
Shape y: (692500,)
Shape X_test_final: (296786, 19)


In [7]:
# ============================================================
# 6. Identificar columnas categóricas para CatBoost
#    (todas las de tipo 'object' excepto la variable objetivo)
# ============================================================
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
print("Columnas categóricas:", cat_cols)

# (Opcional) Convertir explícitamente a string por si hay mezclas raras
for col in cat_cols:
    X[col] = X[col].astype(str)
    X_test_final[col] = X_test_final[col].astype(str)


Columnas categóricas: ['E_PRGM_ACADEMICO', 'E_PRGM_DEPARTAMENTO', 'E_VALORMATRICULAUNIVERSIDAD', 'E_HORASSEMANATRABAJA', 'F_ESTRATOVIVIENDA', 'F_TIENEINTERNET', 'F_EDUCACIONPADRE', 'F_TIENELAVADORA', 'F_TIENEAUTOMOVIL', 'E_PRIVADO_LIBERTAD', 'E_PAGOMATRICULAPROPIO', 'F_TIENECOMPUTADOR', 'F_TIENEINTERNET.1', 'F_EDUCACIONMADRE']


In [11]:
# ============================================================
# SUPER RÁPIDO: sample de filas + CatBoost liviano
# ============================================================

from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import accuracy_score, classification_report

TARGET_COL = 'RENDIMIENTO_GLOBAL'
ID_COL = 'ID'

# 1. Separar features / target
X_full = train.drop(columns=[TARGET_COL, ID_COL])
y_full = train[TARGET_COL]
X_test_final = test.drop(columns=[ID_COL])

print("Shape completo:", X_full.shape, "y:", y_full.shape)

# 2. Columnas categóricas
cat_cols = X_full.select_dtypes(include=['object']).columns.tolist()
print("Categóricas:", cat_cols)

for col in cat_cols:
    X_full[col] = X_full[col].astype(str)
    X_test_final[col] = X_test_final[col].astype(str)

# 3. SAMPLE estratificado para acelerar (ej: 200.000 filas)
TRAIN_ROWS = 200_000   # si aún va rápido, luego puedes subir a 250_000

X_part, _, y_part, _ = train_test_split(
    X_full,
    y_full,
    train_size=TRAIN_ROWS,
    stratify=y_full,
    random_state=42
)

print("Shape sample:", X_part.shape, "y:", y_part.shape)

# 4. Train/valid split sobre el sample
X_train, X_valid, y_train, y_valid = train_test_split(
    X_part,
    y_part,
    test_size=0.2,
    stratify=y_part,
    random_state=42
)

print("X_train:", X_train.shape, "   X_valid:", X_valid.shape)

# 5. Pools de CatBoost
train_pool = Pool(X_train, label=y_train, cat_features=cat_cols)
valid_pool = Pool(X_valid, label=y_valid, cat_features=cat_cols)
test_pool  = Pool(X_test_final, cat_features=cat_cols)

# 6. Modelo CatBoost MUY liviano
model = CatBoostClassifier(
    loss_function='MultiClass',
    eval_metric='Accuracy',
    random_seed=42,
    learning_rate=0.2,       # más alto => menos árboles necesarios
    depth=6,                 # menos profundo => más rápido
    l2_leaf_reg=4,
    iterations=100,          # !!! solo 100 árboles
    bootstrap_type='Bernoulli',
    subsample=0.8,
    rsm=0.8,
    od_type='Iter',
    od_wait=20,
    verbose=10,
    thread_count=-1          # usar todos los cores
)

model.fit(
    train_pool,
    eval_set=valid_pool,
    use_best_model=True
)

# 7. Evaluación rápida en valid
y_valid_pred = model.predict(X_valid).flatten()
acc = accuracy_score(y_valid, y_valid_pred)
print(f"\nAccuracy valid: {acc:.4f}\n")
print(classification_report(y_valid, y_valid_pred))


Shape completo: (692500, 19) y: (692500,)
Categóricas: ['E_PRGM_ACADEMICO', 'E_PRGM_DEPARTAMENTO', 'E_VALORMATRICULAUNIVERSIDAD', 'E_HORASSEMANATRABAJA', 'F_ESTRATOVIVIENDA', 'F_TIENEINTERNET', 'F_EDUCACIONPADRE', 'F_TIENELAVADORA', 'F_TIENEAUTOMOVIL', 'E_PRIVADO_LIBERTAD', 'E_PAGOMATRICULAPROPIO', 'F_TIENECOMPUTADOR', 'F_TIENEINTERNET.1', 'F_EDUCACIONMADRE']
Shape sample: (200000, 19) y: (200000,)
X_train: (160000, 19)    X_valid: (40000, 19)
0:	learn: 0.3684438	test: 0.3703250	best: 0.3703250 (0)	total: 874ms	remaining: 1m 26s
10:	learn: 0.4136813	test: 0.4169750	best: 0.4169750 (10)	total: 9.45s	remaining: 1m 16s
20:	learn: 0.4209188	test: 0.4226750	best: 0.4226750 (20)	total: 16.5s	remaining: 1m 2s
30:	learn: 0.4255875	test: 0.4243500	best: 0.4244750 (29)	total: 25s	remaining: 55.7s
40:	learn: 0.4278875	test: 0.4268750	best: 0.4270250 (39)	total: 38.6s	remaining: 55.5s
50:	learn: 0.4305625	test: 0.4293000	best: 0.4293000 (50)	total: 47.6s	remaining: 45.7s
60:	learn: 0.4326313	test:

In [12]:
# ============================================================
# Generar my_submission.csv para Kaggle
# ============================================================
test_pred = model.predict(X_test_final).flatten()

submission = pd.DataFrame({
    'ID': test[ID_COL],
    'RENDIMIENTO_GLOBAL': test_pred
})

submission.to_csv('my_submission.csv', index=False)
print(submission.head())
print("Guardado my_submission.csv con shape:", submission.shape)


       ID RENDIMIENTO_GLOBAL
0  550236               bajo
1   98545         medio-alto
2  499179               alto
3  782980               bajo
4  785185               bajo
Guardado my_submission.csv con shape: (296786, 2)


In [13]:
# ============================================================
# 13. Enviar submission a Kaggle
#     (Asegúrate de que ya probaste antes que el archivo está bien)
# ============================================================
!kaggle competitions submit -c udea-ai-4-eng-20252-pruebas-saber-pro-colombia -f my_submission.csv -m "versión final catboost"


100% 4.05M/4.05M [00:01<00:00, 3.36MB/s]
Successfully submitted to UDEA/ai4eng 20252 - Pruebas Saber Pro Colombia

In [ ]:
!kaggle competitions submit -c udea-ai-4-eng-20252-pruebas-saber-pro-colombia -f my_submission.csv -m "versión final catboost"